PyAnswerBot

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install transformers datasets trl accelerate beautifulsoup4 pandas -q

In [ ]:
import torch
import transformers
import datasets
import trl
import pandas
from bs4 import BeautifulSoup

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)
print("pandas:", pandas.__version__)
print("GPU available:", torch.cuda.is_available())
print("All libraries ready! ")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = "/content/drive/MyDrive/finetools/"

questions = pd.read_csv(path + "Questions.csv", encoding="latin-1")
answers = pd.read_csv(path + "Answers.csv", encoding="latin-1")
tags = pd.read_csv(path + "Tags.csv", encoding="latin-1")

print("Questions:", questions.shape)
print("Answers:", answers.shape)
print("Tags:", tags.shape)
print("CSVs loaded! ")

In [ ]:
# Keep only high scored questions and answers
questions = questions[questions["Score"] >= 5]
answers = answers[answers["Score"] >= 5]

# Keep only Python questions
python_ids = tags[tags["Tag"] == "python"]["Id"]
questions = questions[questions["Id"].isin(python_ids)]

print("Filtered Questions:", len(questions))
print("Filtered Answers:", len(answers))
print("Filtered! ")

In [ ]:
# Get best answer for each question (highest score)
best_answers = answers.sort_values("Score", ascending=False)\
                       .drop_duplicates(subset="ParentId")

# Merge questions with answers
df = questions.merge(best_answers, left_on="Id", right_on="ParentId")

# Keep only useful columns
df = df[["Title", "Body_x", "Body_y"]].dropna()
df.columns = ["Title", "Question", "Answer"]

print("Total Q&A pairs:", len(df))
print(df.head(2))
print("Merged! ")

In [ ]:
import json
from bs4 import BeautifulSoup

# Clean HTML tags from text
def clean_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

df["Question"] = df["Question"].apply(clean_html)
df["Answer"] = df["Answer"].apply(clean_html)
print("HTML cleaned! ")

# Convert to JSONL format
with open("training_data.jsonl", "w") as f:
    for _, row in df.iterrows():
        example = {
            "messages": [
                {"role": "system", "content": "You are a helpful assistant that answers Python programming questions."},
                {"role": "user", "content": f"{row['Title']}\n\n{row['Question']}"},
                {"role": "assistant", "content": row["Answer"]}
            ]
        }
        f.write(json.dumps(example) + "\n")

print(f"JSONL created! Total examples: {len(df)} ")

Step-4 Loading Modle

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "facebook/opt-125m"  # small & fast model, perfect for Colab

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # uses less GPU memory
    device_map="auto"           # automatically puts model on GPU
)

tokenizer.pad_token = tokenizer.eos_token

print("Model loaded on GPU! ")
print("Device:", next(model.parameters()).device)

Step -5 : Training the Model

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Load JSONL dataset
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")
print("Dataset loaded! Total examples:", len(dataset))

# Format examples as plain text
def format_example(example):
    text = ""
    for msg in example["messages"]:
        text += f"{msg['role']}: {msg['content']}\n"
    return {"text": text}

dataset = dataset.map(format_example)

# Tokenize dataset
tokenized_dataset = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512, padding="max_length"),
    batched=True
)
print("Dataset tokenized! ")

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="./finetuned-model",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        save_steps=100,
        logging_steps=10,
        bf16=False,
        fp16=False,
        gradient_checkpointing=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
    train_dataset=tokenized_dataset,
)

print("Starting training... ")
trainer.train()
print("Training done! ")


In [ ]:
model.save_pretrained("./finetuned-model")
tokenizer.save_pretrained("./finetuned-model")

In [ ]:
import torch

input_text = "user: How do I reverse a list in Python?\nassistant:"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.3,  # prevents repeating
        no_repeat_ngram_size=3,  # prevents repeating phrases
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Only show the answer part
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = full_text.split("assistant:")[-1].strip()
print("Answer:")
print(answer)

## Conclusion

PyAnswerBot is a fine-tuned AI model trained on real Stack Overflow data to answer Python programming questions. The project covered the complete machine learning pipeline -- downloading and cleaning over 4,436 high quality Q&A pairs from Kaggle, removing HTML tags, converting data into JSONL format, and fine-tuning the facebook/opt-125m language model using Hugging Face TRL on a free Tesla T4 GPU in Google Colab.
This project demonstrated that building a domain-specific AI model is possible even with zero budget, using only free tools like Google Colab and Hugging Face. As a first year Data Science student, this hands-on experience covered real world ML skills including data cleaning, preprocessing, GPU training, and model evaluation -- which are the core skills required in any machine learning engineering role today.